In [1]:
# Импорты

import json
import random
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
import sys
import os

d:\DataScience\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ПУТИ

ROOT = Path("..")
PROCESSED_PATH = ROOT / "data" / "processed"
SPLITS_PATH = ROOT / "data" / "splits"
MODELS_PATH = ROOT / "models"
MODELS_PATH.mkdir(exist_ok=True)

In [3]:
from pathlib import Path
sys.path.append(os.path.abspath(".."))

In [4]:
from src.data import (
    collect_dataset,
    AnimeDataset,
)
from src.train import (
    train_epoch,
    validate_epoch,
)

In [6]:
# Загружаем информацию о датасете
dataset = collect_dataset(PROCESSED_PATH)

print("Изображений:", len(dataset))

Изображений: 24009


In [7]:
# Разделение датасета на обучающую, валидационную и тестовую выборки

# Получаем список меток для стратифицированного разделения
labels = [sample["label"] for sample in dataset]

# Разделяем датасет на обучающую (70%) и временную (30%) выборки
train, temp = train_test_split(
    dataset,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

# Получаем метки временной выборки
temp_labels = [sample["label"] for sample in temp]

# Разделяем временную выборку на валидационную (15%) и тестовую (15%)
val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train:", len(train))
print("Val:", len(val))
print("Test:", len(test))

Train: 16806
Val: 3601
Test: 3602


In [8]:
# Сохранение разбиения датасета

with open(SPLITS_PATH / "train.json", "w", encoding="utf-8") as f:
    json.dump(train, f, ensure_ascii=False, indent=4)

with open(SPLITS_PATH / "val.json", "w", encoding="utf-8") as f:
    json.dump(val, f, ensure_ascii=False, indent=4)

with open(SPLITS_PATH / "test.json", "w", encoding="utf-8") as f:
    json.dump(test, f, ensure_ascii=False, indent=4)

print("Разбиение сохранено.")

Разбиение сохранено.


In [10]:
# Преобразования изображений

# Преобразования для обучающей выборки
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Преобразования для валидационной и тестовой выборок
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
CLASSES = sorted({sample["label"] for sample in dataset})

CLASS_TO_IDX = {
    cls: idx
    for idx, cls in enumerate(CLASSES)
}

In [13]:
# Создание датасетов

train_dataset = AnimeDataset(
    dataset=train,
    class_to_idx=CLASS_TO_IDX,
    transform=train_transform,
)

val_dataset = AnimeDataset(
    dataset=val,
    class_to_idx=CLASS_TO_IDX,
    transform=test_transform,
)

test_dataset = AnimeDataset(
    dataset=test,
    class_to_idx=CLASS_TO_IDX,
    transform=test_transform,
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

16806
3601
3602


In [14]:
# Создание DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoader готовы")

DataLoader готовы


In [15]:
# Классы

CLASSES = sorted({
    sample["label"]
    for sample in dataset
})

CLASS_TO_IDX = {
    label: idx
    for idx, label in enumerate(CLASSES)
}

IDX_TO_CLASS = {
    idx: label
    for label, idx in CLASS_TO_IDX.items()
}

print("Классов:", len(CLASSES))

Классов: 23


In [17]:
# Создание модели

# Определяем устройство для вычислений
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Загружаем предобученную модель ResNet50
model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

# Замораживаем параметры backbone
for param in model.parameters():
    param.requires_grad = False

# Заменяем классификационную голову
model.fc = nn.Linear(
    model.fc.in_features,
    len(CLASSES)
)

# Разрешаем обучать только классификатор
for param in model.fc.parameters():
    param.requires_grad = True

# Переносим модель на выбранное устройство
model = model.to(device)

print("Device:", device)

Device: cuda


In [18]:
def count_parameters(model):
    """
    Подсчитывает количество обучаемых параметров модели.

    Args:
        model (nn.Module): Нейронная сеть.

    Returns:
        int: Количество параметров, участвующих в обучении.
    """
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# Выводим количество обучаемых параметров модели
print(f"Trainable parameters: {count_parameters(model):,}")

Trainable parameters: 47,127


In [19]:
# Настройка обучения

# Функция потерь для задачи классификации
criterion = nn.CrossEntropyLoss()

# Оптимизатор для обучения классификационной головы
optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Планировщик изменения скорости обучения
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [22]:
# Параметры обучения

EPOCHS = 20
FREEZE_EPOCHS = 5
PATIENCE = 5

best_acc = 0
patience = 0

# Логирование обучения в TensorBoard
writer = SummaryWriter(
    "../logs/resnet50"
)

In [24]:
# Обучение модели

for epoch in range(EPOCHS):
    # Обучение и валидация
    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )

    val_loss, val_acc = validate_epoch(
        model,
        val_loader,
        criterion,
        device,
    )

    # Обновляем планировщик скорости обучения
    scheduler.step()

    # Сохраняем метрики в TensorBoard
    writer.add_scalar(
        "Loss/train",
        train_loss,
        epoch
    )

    writer.add_scalar(
        "Loss/val",
        val_loss,
        epoch
    )

    writer.add_scalar(
        "Accuracy/train",
        train_acc,
        epoch
    )

    writer.add_scalar(
        "Accuracy/val",
        val_acc,
        epoch
    )

    # Выводим результаты текущей эпохи
    print(f"Epoch {epoch + 1}/{EPOCHS}")

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # Сохраняем модель при улучшении точности
    if val_acc > best_acc:
        best_acc = val_acc
        patience = 3

        torch.save(
            model.state_dict(),
            MODELS_PATH / "best_resnet50.pth"
        )

        print("Best model saved")

    else:
        patience += 1

    # Останавливаем обучение при отсутствии улучшений
    if patience >= PATIENCE:
        print("Early stopping")
        break

writer.close()

100%|██████████| 113/113 [00:32<00:00,  3.50it/s]


Epoch 1/20
Train Loss: 2.1606 | Train Acc: 0.4342
Val Loss: 1.7254 | Val Acc: 0.5612
Best model saved


100%|██████████| 113/113 [00:30<00:00,  3.76it/s]


Epoch 2/20
Train Loss: 1.5207 | Train Acc: 0.6015
Val Loss: 1.4523 | Val Acc: 0.6179
Best model saved


100%|██████████| 113/113 [00:30<00:00,  3.73it/s]


...

Epoch 9/20
Train Loss: 0.9407 | Train Acc: 0.7503
Val Loss: 1.1234 | Val Acc: 0.6931
Best model saved


100%|██████████| 113/113 [00:31<00:00,  3.57it/s]


Epoch 10/20
Train Loss: 0.9270 | Train Acc: 0.7554
Val Loss: 1.1410 | Val Acc: 0.6840


100%|██████████| 113/113 [00:31<00:00,  3.64it/s]

Epoch 11/20
Train Loss: 0.9300 | Train Acc: 0.7549
Val Loss: 1.1399 | Val Acc: 0.6856
Early stopping


In [25]:
# Загрузка лучшей модели

model.load_state_dict(
    torch.load(
        ROOT / "models" / "best_resnet50.pth",
        map_location=device
    )
)

# Переводим модель в режим инференса
model.eval()

C:\Users\maxim\AppData\Local\Temp\ipykernel_34104\682672564.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [27]:
# Оценка качества модели на тестовой выборке

test_loss, test_acc = validate_epoch(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

100%|██████████| 113/113 [00:55<00:00,  2.02it/s]

Test Loss: 1.1084
Test Acc : 0.7057


In [28]:
print(device)
print(next(model.parameters()).device)

cuda
cuda:0
